# AgroUTMACH PIX — NB00
## Configuración inicial y descarga directa del dataset PIO desde Zenodo

Este notebook prepara la estructura del proyecto en Google Drive y descarga los archivos del registro Zenodo **16686320** directamente a Drive, evitando descargar primero al computador.

**Proyecto:** AgroUTMACH PIX  
**Asignatura:** Procesamiento Digital de Imágenes  
**Aplicación:** detección, conteo y seguimiento de pollos con YOLO


### 1. Montar Google Drive
Ejecute esta celda y autorice el acceso a su Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


### 2. Crear/verificar la estructura del proyecto
La celda es segura para volver a ejecutarse: si las carpetas ya existen, no las duplica.


In [ ]:
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/AgroUTMACH_PIX')
FOLDERS = [
    '00_ZENODO_ORIGINAL',
    '01_DATASET_PIO',
    '02_NOTEBOOKS',
    '03_MODELOS',
    '04_VIDEOS_FERIA',
    '05_RESULTADOS',
    '06_DASHBOARD',
    '07_RECURSOS_STAND',
]

ROOT.mkdir(parents=True, exist_ok=True)
for folder in FOLDERS:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

print('✅ Estructura verificada en:', ROOT)
for p in sorted(ROOT.iterdir()):
    if p.is_dir():
        print('  📁', p.name)


### 3. Consultar automáticamente el registro de Zenodo
Esta celda consulta la API oficial de Zenodo y muestra los archivos realmente disponibles en el registro.


In [ ]:
import requests
from pprint import pprint

RECORD_ID = '16686320'
API_URL = f'https://zenodo.org/api/records/{RECORD_ID}'

r = requests.get(API_URL, timeout=60)
r.raise_for_status()
record = r.json()

files = record.get('files', [])
print(f'✅ Registro Zenodo {RECORD_ID}')
print(f'Archivos encontrados: {len(files)}\n')

def gb(n):
    return n / (1024**3)

for i, f in enumerate(files, 1):
    print(f'{i:02d}. {f.get("key")}  |  {gb(f.get("size",0)):.3f} GB')


### 4. Descargar todos los archivos directamente a Drive
Por defecto, el notebook descarga **todos** los archivos del registro dentro de `00_ZENODO_ORIGINAL`.  
Si una descarga ya está completa y el tamaño coincide, se omite automáticamente.

> **Nota:** `Videos 2023.rar` es grande; la descarga puede tardar según la conexión y la velocidad de escritura de Drive.


In [ ]:
import os
from pathlib import Path
from tqdm.auto import tqdm
import requests

DEST = ROOT / '00_ZENODO_ORIGINAL'
DEST.mkdir(parents=True, exist_ok=True)

DOWNLOAD_ALL = True
# Si algún día quiere limitar la descarga, cambie DOWNLOAD_ALL=False
# y escriba aquí los nombres exactos que desee.
SELECTED_FILES = {
    'data.rar',
    'Readme.txt',
    'FilePrefixCode.xlsx',
    'Test 2025.rar',
    'Videos 2023.rar',
    '0002.mp4',
}

def zenodo_download(file_meta, dest_dir):
    name = file_meta['key']
    expected = int(file_meta.get('size', 0))
    dest = Path(dest_dir) / name

    if dest.exists() and expected > 0 and dest.stat().st_size == expected:
        print(f'⏭️  Ya existe y está completo: {name}')
        return dest

    url = file_meta.get('links', {}).get('content') or file_meta.get('links', {}).get('self')
    if not url:
        raise RuntimeError(f'No se encontró enlace de descarga para {name}')

    # Reanuda si quedó un archivo parcial.
    mode = 'wb'
    headers = {}
    initial = 0
    if dest.exists() and expected > 0 and dest.stat().st_size < expected:
        initial = dest.stat().st_size
        headers['Range'] = f'bytes={initial}-'
        mode = 'ab'

    with requests.get(url, stream=True, headers=headers, timeout=(30, 300)) as resp:
        # Si el servidor ignora Range, reiniciamos para no corromper el archivo.
        if initial and resp.status_code == 200:
            initial = 0
            mode = 'wb'
        resp.raise_for_status()

        total = expected if expected > 0 else int(resp.headers.get('content-length', 0)) + initial
        with open(dest, mode) as fh, tqdm(
            total=total, initial=initial, unit='B', unit_scale=True, unit_divisor=1024,
            desc=name, dynamic_ncols=True
        ) as bar:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    fh.write(chunk)
                    bar.update(len(chunk))

    if expected > 0 and dest.stat().st_size != expected:
        raise IOError(f'Tamaño incompleto en {name}: {dest.stat().st_size} != {expected}')

    print(f'✅ Descargado: {name}')
    return dest

queue = files if DOWNLOAD_ALL else [f for f in files if f.get('key') in SELECTED_FILES]
print(f'Archivos a descargar: {len(queue)}')
for f in queue:
    zenodo_download(f, DEST)

print('\n✅ Descarga finalizada.')


### 5. Verificar lo descargado
Muestra el tamaño de cada archivo guardado en Drive.


In [ ]:
from pathlib import Path

print('Contenido de 00_ZENODO_ORIGINAL:\n')
for p in sorted(DEST.iterdir()):
    if p.is_file():
        print(f'{p.name:28s}  {p.stat().st_size/(1024**3):8.3f} GB')


### 6. Verificar checksums MD5 cuando Zenodo los proporciona
Esta comprobación es opcional, pero sirve para confirmar que los archivos grandes no se corrompieron durante la descarga.


In [ ]:
import hashlib

def md5sum(path, chunk=16*1024*1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            data = f.read(chunk)
            if not data:
                break
            h.update(data)
    return h.hexdigest()

for f in files:
    name = f.get('key')
    checksum = f.get('checksum', '')
    path = DEST / name
    if not path.exists() or not checksum.startswith('md5:'):
        continue
    expected = checksum.split(':', 1)[1]
    actual = md5sum(path)
    status = '✅ OK' if actual == expected else '❌ NO COINCIDE'
    print(f'{status}  {name}')


### 7. Próximo paso
Cuando todos los archivos estén descargados y verificados, continuaremos con **NB01 — Auditoría y preparación del dataset PIO para YOLO**.  
En ese notebook revisaremos:

- estructura `images/` y `labels/`;
- correspondencia imagen–anotación;
- clases del dataset;
- ejemplos de cajas YOLO;
- distribución train/val;
- selección del material que usaremos en la feria.
